In [ ]:
output_dir = "../outputs"

In [ ]:
import sys
sys.path.append("..")

import os
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib as mpl
from omegaconf import OmegaConf
from pprint import pp
import ipywidgets as widgets
from IPython.display import display, clear_output

from src.utils.visualize import mask_to_alpha_image

In [ ]:
run_groups = os.listdir(output_dir)
run_groups.sort()

def make_run_selectors(change):
    with run_selectors_out:
        all_runs = {group: [run for run in os.listdir(os.path.join(output_dir, group)) if os.path.exists(os.path.join(output_dir, group, run, "results.pickle"))] for group in change["new"]}
        for runs in all_runs.values():
            runs.sort()
        run_selectors.clear()
        for group, runs in all_runs.items():
            run_selectors[group] = widgets.SelectMultiple(options=runs, value=runs, description=f"{group}:")
        run_selectors_out.clear_output(wait=True)
        for selector in run_selectors.values():
            selector.layout.height = f"{8+len(selector.options)*19}px"
        display(widgets.HBox(list(run_selectors.values())))

run_group_selector = widgets.SelectMultiple(options=run_groups, description="Select run groups:")
run_group_selector.layout.height = f"{20+len(run_groups)*19}px"
run_selectors = {}
run_group_selector.observe(make_run_selectors, names="value")

display(run_group_selector)
run_selectors_out = widgets.Output()
run_selectors_out

In [ ]:
def flatten_config(conf):
    dict_ = OmegaConf.to_container(conf, resolve=True)
    return flatten_config_helper(dict_)

def flatten_config_helper(original, prefix=""):
    result = {}
    for key, val in original.items():
        if isinstance(val, dict):
            result = result | flatten_config_helper(val, prefix + key + ".")
        elif isinstance(val, list):
            if len(val)>0 and isinstance(val[0], dict): # flatten dicts to make hashable
                for dict_ in val:
                    result = result | flatten_config_helper(dict_, prefix + key + ".")
            else: # convert to tuple to make it hashable
                result[prefix + key] = tuple(val)
        else:
            result[prefix + key] = val
    return result

def config_diff(configs):
    sets = [set(conf.items()) for conf in configs]
    first_set = sets[0]
    set_diff = set()
    for set_ in sets[1:]:
        set_diff = set_diff.union(first_set ^ set_)
    return list(dict(set_diff).keys())

runs = []
run_paths = []
for group, selector in run_selectors.items():
    runs.extend([run for run in selector.value])
    run_paths.extend([os.path.join(output_dir, group, run) for run in selector.value])

config_list = [OmegaConf.load(os.path.join(path, ".hydra/config.yaml")) for path in run_paths]
for i, conf in enumerate(config_list):
    conf.run_name = runs[i]

load_images_selector = widgets.Checkbox(
    description='Load images: ',
    value = False,
)

if len(config_list) > 1:
    flattened_configs = [flatten_config(conf) for conf in config_list]
    config_diff_keys = config_diff(flattened_configs)
    conf_selector = widgets.SelectMultiple(
        options=config_diff_keys,
        value=config_diff_keys,
        description='Select config parameters to include:',
        )
else:
    flattened_configs = [flatten_config(config_list[0])]
    config_diff_keys = []
    conf_selector = widgets.SelectMultiple(
        options=[],
        value=[],
        description='Select config parameters to include:',
        )
conf_selector.layout.height = f"{8+len(conf_selector.options)*19}px"
display(conf_selector)
display(load_images_selector)

In [ ]:
shared_config = {key: val for key, val in flattened_configs[0].items() if key not in config_diff_keys}
selected_config = [{key: val for key, val in conf.items() if key in conf_selector.value} for conf in flattened_configs]
non_selected_config = [{key: val for key, val in conf.items() if key in config_diff_keys and key not in conf_selector.value} for conf in flattened_configs]
results_list = [pd.read_pickle(os.path.join(path, "results.pickle")) for path in run_paths]
for i, results in enumerate(results_list):
    if load_images_selector.value:
        conf = flattened_configs[i]
        image_path = os.path.join(output_dir, conf["run_group"], conf["run_name"], "images")
        results["target_image"] = results["opencanopy_idx"].apply(lambda idx: torch.load(os.path.join(image_path, str(idx)) + "_target_image.pt"))
        results["reconstructed_target"] = results["opencanopy_idx"].apply(lambda idx: torch.load(os.path.join(image_path, str(idx)) + "_reconstructed_target.pt"))
        results["test_mask"] = results["opencanopy_idx"].apply(lambda idx: torch.load(os.path.join(image_path, str(idx)) + "_test_mask.pt"))
        results["train_mask"] = results["opencanopy_idx"].apply(lambda idx: torch.load(os.path.join(image_path, str(idx)) + "_train_mask.pt"))
    for name, val in selected_config[i].items():
        results[name] = pd.Series([val for _ in range(len(results))])
all_results = pd.concat(results_list)
all_results = all_results.fillna("nan")
print("Shared config:")
pp(shared_config)
print("\nSelected parameters:")
pp(selected_config)
print("\nNon-selected parameters:")
pp(non_selected_config)

available_metrics = [column_name[10:] for column_name in all_results.columns if column_name[:10] == "all_train_"]
available_full_image_metrics = [column_name[4:] for column_name in all_results.columns if column_name[:14] == "all_full_image"]

chosen_conf = list(conf_selector.value)

#######################################################################
# Source - https://stackoverflow.com/a/34162576
# Posted by DSM
# Retrieved 2026-04-05, License - CC BY-SA 3.0

def query_df(df, dict_):
    return df.loc[(df[list(dict_)] == pd.Series(dict_)).all(axis=1)]
    
#######################################################################

# Filtering

In [ ]:
# Filtering results (easier than selecting runs)
indices = [33650, 6131, 32747, 11829]
sparsity_omega = {1.0: 32.0, 0.5: 16.0, 0.1: 8.0, 0.02: 4.0}
mask1 = (all_results["data.sparsity"] == 1.0) & (all_results["model.first_omega_0"] == sparsity_omega[1.0])
mask2 = (all_results["data.sparsity"] == 0.5) & (all_results["model.first_omega_0"] == sparsity_omega[0.5])
mask3 = (all_results["data.sparsity"] == 0.1) & (all_results["model.first_omega_0"] == sparsity_omega[0.1])
mask4 = (all_results["data.sparsity"] == 0.02) & (all_results["model.first_omega_0"] == sparsity_omega[0.02])
all_results = all_results[(mask1 | mask2 | mask3 | mask4) & all_results["opencanopy_idx"].isin(indices)]

chosen_conf.remove("model.first_omega_0")

In [ ]:
# Filtering results (easier than selecting runs)
indices = [33650, 6131, 32747, 11829]
sparsity_scale = {1.0: 8.0, 0.5: 4.0, 0.1: 2.0, 0.02: 0.5}
mask1 = (all_results["data.sparsity"] == 1.0) & (all_results["model.scale"] == sparsity_scale[1.0])
mask2 = (all_results["data.sparsity"] == 0.5) & (all_results["model.scale"] == sparsity_scale[0.5])
mask3 = (all_results["data.sparsity"] == 0.1) & (all_results["model.scale"] == sparsity_scale[0.1])
mask4 = (all_results["data.sparsity"] == 0.02) & (all_results["model.scale"] == sparsity_scale[0.02])
all_results = all_results[(mask1 | mask2 | mask3 | mask4) & all_results["opencanopy_idx"].isin(indices)]

chosen_conf.remove("model.scale")

In [ ]:
mask1 = all_results["model.scale"] != 0.25
mask2 = all_results["model.scale"] != 1.0
mask3 = all_results["model.scale"] != 16.0
all_results = all_results[mask1 & mask2 & mask3]

In [ ]:
mask1 = all_results["model.first_omega_0"] != 2.0
mask2 = all_results["model.first_omega_0"] != 64.0
all_results = all_results[mask1 & mask2]

# Single parameter combintion inspection

In [ ]:
def make_metric_plot(all_results, query_dict, metric, log_scale):
    fig, ax = plt.subplots()
    query_result = query_df(all_results, query_dict)
    if len(query_result) > 1:
        raise ValueError("Not unique combination!")
    elif len(query_result) == 0:
        raise ValueError("No run satisfies combination!")
    else:
        if metric[:10] == "full_image":
            ax.plot(query_result[f"all_{metric}"].item(), label=metric)
        else:
            ax.plot(query_result[f"all_train_{metric}"].item(), label=f"train_{metric}")
            ax.plot(query_result[f"all_test_{metric}"].item(), label=f"test_{metric}")
        if log_scale:
            ax.set_yscale("log")
        ax.legend()
        return fig

def make_image_plot(all_results, query_dict, cmap="summer"):
    query_result = query_df(all_results, query_dict)
    if len(query_result) > 1:
        raise ValueError("Not unique combination!")
    elif len(query_result) == 0:
        raise ValueError("No run satisfies combination!")
    else:
        target_image = query_result["target_image"].item()
        reconstructed = query_result["reconstructed_target"].item()
        pixel_min = min(target_image.min(), reconstructed.min())
        pixel_max = max(target_image.max(), reconstructed.max())
        fig, ax = plt.subplots(3, 2, figsize=(12,18))
        for i in range(3):
            imshow = ax[i, 0].imshow(target_image, vmin=pixel_min, vmax=pixel_max, cmap=cmap)
            fig.colorbar(imshow, ax=ax[i,0])
            ax[i, 0].set_title("target image")
        
        ax[0, 1].imshow(target_image, vmin=pixel_min, vmax=pixel_max, cmap=cmap)
        mask = query_result["train_mask"].item()
        ax[0, 1].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
        ax[0, 1].set_title("sparse (train)")
    
        ax[1, 1].imshow(reconstructed, vmin=pixel_min, vmax=pixel_max, cmap=cmap)
        ax[1, 1].set_title("reconstructed")
    
        ax[2, 1].imshow(target_image, vmin=pixel_min, vmax=pixel_max, cmap=cmap)
        mask = query_result["test_mask"].item()
        ax[2, 1].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
        ax[2, 1].set_title("test")
        for something in ax:
            for this in something:
                this.set_axis_off()
        return fig

def make_plots(change):
    with out:
        out.clear_output(wait=True)
        chosen_options = {list(param_dict)[i]: sel.value for i, sel in enumerate(param_selectors)}
        chosen_options["opencanopy_idx"] = opencanopy_idx_selector.value
        fig_metric = make_metric_plot(all_results=all_results, query_dict=chosen_options, metric=metric_selector.value, log_scale=log_scale_selector.value)
        fig_image = make_image_plot(all_results=all_results, query_dict=chosen_options)
        display(fig_metric)
        display(fig_image)

param_dict = {key: all_results[key].unique() for key in chosen_conf}

param_selectors = [widgets.ToggleButtons(options=vals, description=key+":") for key, vals in param_dict.items()]
opencanopy_idx_selector = widgets.Dropdown(
    options=all_results["opencanopy_idx"].unique(),
    description='opencanopy idx:',
)
metric_selector = widgets.Dropdown(
    options=available_metrics+available_full_image_metrics,
    value='loss',
    description='metric:',
)
log_scale_selector = widgets.Checkbox(
    value=True,
    description='log scale:',
)

for selector in param_selectors:
    selector.observe(make_plots, names="value")
metric_selector.observe(make_plots, names="value")
log_scale_selector.observe(make_plots, names="value")
opencanopy_idx_selector.observe(make_plots, names="value")


for selector in param_selectors:
    display(selector)
print("--------------------------------------------------------------------------------")
display(widgets.HBox([metric_selector, log_scale_selector]))
display(opencanopy_idx_selector)

out = widgets.Output()
out

# Grid

In [ ]:
# let us assume all combinations of chosen parameters are present
def make_image_grid(all_results, x_var, y_var, other_params, flip_x, flip_y, image_name, mask_name, metric, log_scale, color_by, save_fig, save_name, sparsity_row):
    assert (image_name is None and metric is not None) or (image_name is not None and metric is None)
    assert (sparsity_row and x_var == "data.sparsity" and y_var != "opencanopy_idx") or (not sparsity_row)
    width = 6.27
    scale_heights = 10
    height_unit = "m"
    scale_metrics = {"RMSE": 10}
    string_map = {"model.first_omega_0": r"$\omega_0$", "data.sparsity": "sparsity", "model.scale": "scale", "opencanopy_idx": "index", "data.normalization_scale": "norm"}
    metric_unit_string_map = {"RMSE": "RMSE [m]"}
    norm_method_map = {"1.0": "No norm", "85.86": "global std", "600.0": "global min/max", "individual_min_max": "ind min max", "individual_std": "ind std"}

    colorbar_label = metric_unit_string_map[color_by] if color_by in metric_unit_string_map else color_by
    y_var_name = string_map[y_var] if y_var in string_map else y_var
    x_var_name = string_map[x_var] if x_var in string_map else x_var
    colorby_scale = scale_metrics[color_by] if color_by in scale_metrics else 1
    if metric is not None:
        metric_scale = scale_metrics[metric] if metric in scale_metrics else 1
    
    all_results = query_df(all_results, other_params)
    try:
        x_vals = all_results[x_var].sort_values(ascending=flip_x).unique()
    except TypeError:
        all_results[x_var] = all_results[x_var].astype(str)
        x_vals = all_results[x_var].sort_values(ascending=flip_x).unique()
    try:
        y_vals = all_results[y_var].sort_values(ascending=flip_y).unique()
    except:
        all_results[y_var] = all_results[y_var].astype(str)
        y_vals = all_results[y_var].sort_values(ascending=flip_y).unique()
    x_len = len(x_vals)
    y_len = len(y_vals)
    
    min_metric_val = float("inf")
    max_metric_val = float("-inf")
    for y_val in y_vals:
        for x_val in x_vals:
            query_result = query_df(all_results, {x_var: x_val, y_var: y_val})
            assert len(query_result) == 1, f"Not unique combination (length: {len(query_result)}) ({x_var}: {x_val}) ({y_var}: {y_val})"
            if metric is not None:
                if metric[:10] == "full_image":
                    full_image_metrics = query_result[f"all_{metric}"].item()
                    full_image_min = full_image_metrics.min()
                    full_image_max = full_image_metrics.max()
                    if full_image_max > max_metric_val:
                        max_metric_val = full_image_max
                    if full_image_min < min_metric_val:
                        min_metric_val = full_image_min
                else:
                    train_metrics = query_result[f"all_train_{metric}"].item()
                    test_metrics = query_result[f"all_test_{metric}"].item()
                    train_max = train_metrics.max()
                    train_min = train_metrics.min()
                    test_max = test_metrics.max()
                    test_min = test_metrics.min()
                    if train_max > max_metric_val:
                        max_metric_val = train_max
                    if train_min < min_metric_val:
                        min_metric_val = train_min
                    if test_max > max_metric_val:
                        max_metric_val = test_max
                    if test_min < min_metric_val:
                        min_metric_val = test_min

    if metric == "R2" or metric == "full_image_SSIM":
        min_metric_val = 0.0
        max_metric_val = 1.1

    if metric is not None:
        min_metric_val /= metric_scale
        max_metric_val /= metric_scale
    
    color_by_cmap = mpl.colormaps["plasma"]
    color_by_metrics = all_results[f"test_{color_by}"]
    color_by_max = color_by_metrics.max() / colorby_scale
    color_by_min = color_by_metrics.min() / colorby_scale
    color_by_norm = mpl.colors.Normalize(vmin=color_by_min, vmax=color_by_max)
    scalar_mappable = mpl.cm.ScalarMappable(color_by_norm, color_by_cmap)
    if metric is None:
        height = width * (y_len + int(sparsity_row)) / (x_len + 1 + 1/10*2)
        fig, ax = plt.subplots(y_len + int(sparsity_row), x_len + 3, figsize=(width, height), constrained_layout=True, gridspec_kw={'width_ratios': [1, 20] + [20]*x_len + [1]})
        #fig, ax = plt.subplots(y_len + int(sparsity_row), x_len + 3, figsize=(size * x_len + size + size/10*2, size*(y_len + int(sparsity_row))), constrained_layout=True, gridspec_kw={'width_ratios': [1, 20] + [20]*x_len + [1]})
    else:
        #height = width * y_len/x_len
        #fig, ax = plt.subplots(y_len, x_len, figsize=(width, height), constrained_layout=True)
        size = 4
        fig, ax = plt.subplots(y_len, x_len, figsize=(size*x_len, size*y_len), constrained_layout=True)
    if sparsity_row:
        target = all_results["target_image"].iloc[0] / scale_heights
        min_pixel_val = target.min() # scaling is done above
        max_pixel_val = target.max() # scaling is done above
        ax[0, 1].imshow(target, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
        ax[0, 1].set_xticks([], [])
        ax[0, 1].set_yticks([], [])
        ax[0, 1].set_title("Target image", fontsize=8)
        ax[0, 0].axis('off')
        ax[0, -1].axis('off')
        for j, x_val in enumerate(x_vals):
            query_result = query_df(all_results, {x_var: x_val})
            ax[0, j+2].imshow(target, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
            mask = query_result["train_mask"].iloc[0]
            ax[0, j+2].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
            ax[0, j+2].set_xticks([], [])
            ax[0, j+2].set_yticks([], [])
            ax[0, j+2].set_title(f"{x_var_name}={x_val}", fontsize=8)
    for i, y_val in enumerate(y_vals):
        if sparsity_row:
            i += 1
        if metric is None:
            target_reference = query_df(all_results, {y_var: y_val})["target_image"].iloc[0] / scale_heights
            min_pixel_val = target_reference.min() # scaling is done above
            max_pixel_val = target_reference.max() # scaling is done above
            target_imshow = ax[i, 1].imshow(target_reference, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
            ax[i, 1].set_xticks([], [])
            ax[i, 1].set_yticks([], [])
            ax[0, 1].set_title("Target image", fontsize=8)
            locator1 = mpl.ticker.LinearLocator(3)
            locator2 = mpl.ticker.LinearLocator(3)
            cb1 = fig.colorbar(target_imshow, cax=ax[i,0], location="left", label=f"height [{height_unit}]", format=mpl.ticker.FormatStrFormatter('%.1f'))
            cb2 = fig.colorbar(scalar_mappable, cax=ax[i,-1], label=colorbar_label, format=mpl.ticker.FormatStrFormatter('%.1f'))
            cb1.locator = locator1
            cb2.locator = locator2
            cb1.update_ticks()
            cb2.update_ticks()
        for j, x_val in enumerate(x_vals):
            query_result = query_df(all_results, {x_var: x_val, y_var: y_val})
            assert len(query_result) == 1, f"Not unique combination (length: {len(query_result)}) ({x_var}: {x_val}) ({y_var}: {y_val})"
            if metric is not None:
                if metric[:10] == "full_image":
                    ax[i, j].plot(query_result[f"all_{metric}"].item() / metric_scale, label=" ".join(metric.split("_")))
                else:
                    ax[i, j].plot(query_result[f"all_train_{metric}"].item() / metric_scale, label=f"train")
                    ax[i, j].plot(query_result[f"all_test_{metric}"].item() / metric_scale, label=f"test")
                ax[i, j].set_ylim(min_metric_val, max_metric_val)
                ax[i, j].grid(which="both")
                if log_scale:
                    ax[i, j].set_yscale("log")
                ax[i, j].set_xlabel("epochs")
                ax[i, j].set_ylabel(metric_unit_string_map[metric] if metric in metric_unit_string_map else metric)
                ax[i, j].legend()
            else:
                j += 2
                img = query_result[image_name].item() / scale_heights
                ax[i, j].imshow(img, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
                if mask_name is not None:
                    mask = query_result[mask_name].item()
                    ax[i, j].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
                ax[i, j].set_xticks([], [])
                ax[i, j].set_yticks([], [])
                loss_value = query_result[f"test_{color_by}"].item() / colorby_scale
                ax[i, j].patch.set_edgecolor(scalar_mappable.to_rgba(loss_value))
                ax[i, j].patch.set_linewidth(4)
                #if j == 3:
                    #ax[i, j].set_title(f"<----------- {y_var_name} = {y_val} ----------->", fontsize=8)
                #if j == 2:
                    #ax[i, j].set_ylabel(f"{y_var_name} = {y_val}", fontsize=8)
                if i == 0:
                    #ax[i, j].set_title(f"{x_var_name} = {x_val}", fontsize=8)
                    #ax[i, j].set_title(f"{x_var_name} = {x_val}\n$\omega_0$ = {sparsity_omega[x_val]}", fontsize=8)
                    #ax[i, j].set_title(f"{x_var_name} = {x_val}\nscale = {sparsity_scale[x_val]}", fontsize=8)
                    ax[i, j].set_title(f"{norm_method_map[x_val]}", fontsize=8)
                else:
                    ax[i, j].set_title(f"", fontsize=8)
                #ax[i, j].set_title(f"{y_var_name} = {y_val}, {x_var_name} = {x_val}", fontsize=10)
    if metric is None:
        fig.canvas.draw()
        reference_x1, _, width, _ = ax[0, 1].get_position().bounds
        reference_x2, _, _, _ = ax[0, 2].get_position().bounds
        line_x = (reference_x1 + width + 0.005)
        line = plt.Line2D(
            (line_x, line_x),
            (0, 1),
            color='gray',
            alpha=0.8,
            linewidth=1,
            transform=fig.transFigure,
        )
        fig.add_artist(line)
        if sparsity_row:
            _, reference_y1, _, height = ax[0, 1].get_position().bounds
            _, reference_y2, _, _ = ax[1, 1].get_position().bounds
            line_y = (reference_y1 - 0.007)
            line = plt.Line2D(
                (0, 1),
                (line_y, line_y),
                color='gray',
                alpha=0.8,
                linewidth=1,
                transform=fig.transFigure,
            )
            fig.add_artist(line)
    if save_fig:
        plt.savefig(save_name, dpi=250)
    return fig

def update_param_widgets(update):
    with out2:
        out2.clear_output(wait=True)
        left_over = list(y_var_selector.options)
        left_over.remove(x_var_selector.value)
        left_over.remove(y_var_selector.value)
        param_selectors.clear()
        for param in left_over:
            options = all_results[param].unique()
            param_selectors[param] = widgets.ToggleButtons(options=options, description=f"{param}:")
        for selector in param_selectors.values():
            display(selector)
        if x_var_selector.value == "data.sparsity" and y_var_selector.value != "opencanopy_idx":
            display(sparsity_row_selector)
        else:
            sparsity_row_selector.value = False

image_selector = widgets.ToggleButtons(
    options=["target", "sparse", "reconstructed", "test_grid", "metric"],
    description="show:",
    value="reconstructed"
)
metric_selector_grid = widgets.Dropdown(
    options=available_metrics+available_full_image_metrics,
    description="metric:",
    value="RMSE"
)
log_scale_selector_grid = widgets.Checkbox(
    description="log scale",
    value=True
)
x_var_selector = widgets.ToggleButtons(
    options=list(chosen_conf),
    description="x_var:",
#    value= "data.sparsity" if "data.sparsity" in list(chosen_conf) else None
)
y_var_selector = widgets.ToggleButtons(
    options=list(chosen_conf) + ["opencanopy_idx"],
    description="y_var:"
)
x_flip = widgets.Checkbox(
    description="flip x-axis:"
)
y_flip = widgets.Checkbox(
    description="flip y-axis:"
)
save_fig = widgets.Checkbox(
    description="Save figure:"
)
save_name = widgets.Text(
    placeholder="figure name"
)
sparsity_row_selector = widgets.Checkbox(
    description="Sparsity in first row:",
    value=False
)

x_var_selector.observe(update_param_widgets, names="value")
y_var_selector.observe(update_param_widgets, names="value")

display(widgets.HBox([x_var_selector, x_flip]))
display(widgets.HBox([y_var_selector, y_flip]))
display(widgets.HBox([image_selector, metric_selector_grid, log_scale_selector_grid]))

color_by_selector = widgets.Dropdown(
    options=available_metrics,
    value="RMSE",
    description="color by:"
)
display(color_by_selector)

display(widgets.HBox([save_fig, save_name]))

param_selectors = {}

out2 = widgets.Output()
out2

In [ ]:
match image_selector.value:
    case "target":
        image_column = "target_image"
        mask_column = None
        metric = None
    case "sparse":
        image_column = "target_image"
        mask_column = "train_mask"
        metric = None
    case "reconstructed":
        image_column = "reconstructed_target"
        mask_column = None
        metric = None
    case "test_grid":
        image_column = "target_image"
        mask_column = "test_mask"
        metric = None
    case "metric":
        image_column = None
        mask_column = None
        metric = metric_selector_grid.value
other_params = {name: selector.value for name, selector in param_selectors.items()}
grid_fig = make_image_grid(all_results.copy(), x_var_selector.value, y_var_selector.value, other_params, x_flip.value, y_flip.value, image_column, mask_column, metric, log_scale_selector_grid.value, color_by_selector.value, save_fig.value, save_name.value, sparsity_row_selector.value)

# Dataframe manipulations

In [ ]:
metric_columns = [f"train_{metric}" for metric in available_metrics] + [f"test_{metric}" for metric in available_metrics] + available_full_image_metrics
relevant_columns = list(chosen_conf) + ["opencanopy_idx"] + metric_columns
final_metrics = all_results[relevant_columns]
final_metrics

## Muon FF

In [ ]:
final_metrics.groupby(["data.sparsity", "optimizer._target_"]).mean()[["full_image_SSIM", "test_RMSE", "test_ME"]]#.sort_values(["data.sparsity", "test_RMSE"], ascending=True)

In [ ]:
final_metrics.sort_values("test_RMSE", ascending=True).groupby(["data.sparsity", "opencanopy_idx"]).head(1).groupby(["data.sparsity", "optimizer._target_"]).size()

## Fourier Features scale tuning

In [ ]:
# R2 breaks down on all 0s target because of divison by zer0
final_metrics.groupby(["data.sparsity", "model.scale"]).mean().sort_values(["data.sparsity", "test_RMSE"], ascending=True)[["test_RMSE", "test_MAE", "full_image_SSIM", "test_R2", "test_ME"]]

In [ ]:
final_metrics.groupby(["data.sparsity", "model.scale"]).mean().sort_values(["data.sparsity", "full_image_SSIM"], ascending=False)[["test_RMSE", "full_image_SSIM", "test_R2", "test_ME"]]

In [ ]:
final_metrics.sort_values("test_RMSE").groupby(["data.sparsity", "opencanopy_idx"]).head(1).groupby(["data.sparsity", "model.scale"]).size()

## Sensitivity heatmaps

In [ ]:
# FF heatmap
mean_metrics = FF_results.groupby(["data.sparsity", "model.scale"], as_index=False).mean()
sparsities = mean_metrics["data.sparsity"].sort_values(ascending=False).unique()
scales = mean_metrics["model.scale"].sort_values(ascending=False).unique()

rmse_map_FF = np.zeros((len(scales), len(sparsities)))
ssim_map_FF = np.zeros((len(scales), len(sparsities)))
me_map_FF = np.zeros((len(scales), len(sparsities)))
r2_map_FF = np.zeros((len(scales), len(sparsities)))

for i, scale in enumerate(scales):
    for j, sparsity in enumerate(sparsities):
        rmse = query_df(mean_metrics, {"data.sparsity": sparsity, "model.scale": scale})["test_RMSE"].item() / 10
        ssim = query_df(mean_metrics, {"data.sparsity": sparsity, "model.scale": scale})["full_image_SSIM"].item()
        me = query_df(mean_metrics, {"data.sparsity": sparsity, "model.scale": scale})["test_ME"].item() / 10
        r2 = query_df(mean_metrics, {"data.sparsity": sparsity, "model.scale": scale})["test_R2"].item()
        rmse_map_FF[i, j] = rmse
        ssim_map_FF[i, j] = ssim
        me_map_FF[i, j] = me
        r2_map_FF[i, j] = r2

In [ ]:
# SIREN heatmap
mean_metrics = siren_results.groupby(["data.sparsity", "model.first_omega_0"], as_index=False).mean()
omegas = mean_metrics["model.first_omega_0"].sort_values(ascending=False).unique()

rmse_map_S = np.zeros((len(omegas), len(sparsities)))
ssim_map_S = np.zeros((len(omegas), len(sparsities)))
me_map_S = np.zeros((len(omegas), len(sparsities)))
r2_map_S = np.zeros((len(omegas), len(sparsities)))

for i, omega in enumerate(omegas):
    for j, sparsity in enumerate(sparsities):
        rmse = query_df(mean_metrics, {"data.sparsity": sparsity, "model.first_omega_0": omega})["test_RMSE"].item() / 10
        ssim = query_df(mean_metrics, {"data.sparsity": sparsity, "model.first_omega_0": omega})["full_image_SSIM"].item()
        me = query_df(mean_metrics, {"data.sparsity": sparsity, "model.first_omega_0": omega})["test_ME"].item() / 10
        r2 = query_df(mean_metrics, {"data.sparsity": sparsity, "model.first_omega_0": omega})["test_R2"].item()
        rmse_map_S[i, j] = rmse
        ssim_map_S[i, j] = ssim
        me_map_S[i, j] = me
        r2_map_S[i, j] = r2

In [ ]:
method = ["FF", "SIREN"]
titles = ["RMSE [m]", "SSIM", r"$R^2$", "ME [m]"]
data = [[rmse_map_FF, ssim_map_FF, r2_map_FF, me_map_FF], [rmse_map_S, ssim_map_S, r2_map_S, me_map_S]]

cmap = "hl"
cmaps = [mpl.colormaps["plasma"], mpl.colormaps["plasma"], mpl.colormaps["plasma"], mpl.colormaps["berlin"]]
scalar_maps = []
for i in range(4):
    all_data = np.vstack([data[0][i], data[1][i]])
    norm = mpl.colors.Normalize(vmin=all_data.min(), vmax=all_data.max())
    scalar_mappable = mpl.cm.ScalarMappable(norm, cmaps[i])
    scalar_maps.append(scalar_mappable)

# creating heatmaps manually as colorizer parameter in matplotlibs imshow doesn't seem to work
heatmaps = [[], []]
for i in range(2):
    for j in range(4):
        rows, cols = data[i][j].shape
        heatmap = np.zeros((rows, cols, 4))
        for k in range(rows):
            for l in range(cols):
                heatmap[k, l] = scalar_maps[j].to_rgba(data[i][j][k, l])
        heatmaps[i].append(heatmap)

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(5.5, 7.6), constrained_layout=True, gridspec_kw={'width_ratios': [20, 20, 1]})

for i in range(3):
    for j in range(3):
        if j == 2:
            locator = mpl.ticker.LinearLocator(3)
            cb = fig.colorbar(scalar_maps[i], cax=ax[i,j], location="right", format=mpl.ticker.FormatStrFormatter('%.2f'))
            cb.locator = locator
            cb.update_ticks()
        else:
            if j == 1:
                yaxis = omegas
            else:
                yaxis = scales
            ax[i, j].imshow(heatmaps[j][i])
            for k in range(len(yaxis)):
                for l in range(len(sparsities)):
                    ax[i, j].text(l, k, f"\\small{{{data[j][i][k, l].round(2)}}}", ha="center", va="center", color="white")
            if i == 2:
                ax[i, j].set_xlabel("sparsity")
            if j == 0:
                ax[i, j].set_ylabel(titles[i] + "\n\n" + "scale")
            elif j==1:
                ax[i, j].set_ylabel(r"$\omega_0$")
            if i == 0:
                ax[i, j].set_title(method[j])
            ax[i, j].set_xticks(range(len(sparsities)), labels=sparsities)
            ax[i, j].set_yticks(range(len(yaxis)), labels=yaxis)

plt.savefig("figures/sensitivity_heatmap1/sensitivity_heatmap1.pgf")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(5.5, 2.8), constrained_layout=True, gridspec_kw={'width_ratios': [20, 20, 1]})

i = 3
for j in range(3):
    if j == 2:
        locator = mpl.ticker.LinearLocator(3)
        cb = fig.colorbar(scalar_maps[i], cax=ax[j], location="right", format=mpl.ticker.FormatStrFormatter('%.2f'))
        cb.locator = locator
        cb.update_ticks()
    else:
        if j == 1:
            yaxis = omegas
        else:
            yaxis = scales
        ax[j].imshow(heatmaps[j][i])
        for k in range(len(yaxis)):
            for l in range(len(sparsities)):
                ax[j].text(l, k, f"\\small{{{data[j][i][k, l].round(2)}}}", ha="center", va="center", color="white")
        if i == 3:
            ax[j].set_xlabel("sparsity")
        if j == 0:
            ax[j].set_ylabel(titles[i] + "\n\n" + "scale")
        elif j==1:
            ax[j].set_ylabel(r"$\omega_0$")
        if i == 3:
            ax[j].set_title(method[j])
        ax[j].set_xticks(range(len(sparsities)), labels=sparsities)
        ax[j].set_yticks(range(len(yaxis)), labels=yaxis)

plt.savefig("figures/sensitivity_heatmap2/sensitivity_heatmap2.pgf")

In [ ]:
special_tiles = np.loadtxt("data_management/normalization_low_high.csv")
special_metrics = final_metrics[final_metrics.opencanopy_idx.isin(special_tiles)]
special_metrics.sort_values(["opencanopy_idx", "data.normalization_scale", "model.scale"])[["opencanopy_idx", "data.normalization_scale", "model.scale", "test_RMSE"]].iloc[75:100]

## SIREN tuning

In [ ]:
samples = [14625, 8355, 50345, 26745, 44348, 18381]
final_metrics[final_metrics.opencanopy_idx.isin(samples)].groupby(["data.sparsity", "model.first_omega_0", "optimizer.lr"]).mean()["test_RMSE"]

In [ ]:
final_metrics.groupby(["data.sparsity", "model.first_omega_0"]).mean().sort_values(["data.sparsity", "test_RMSE"], ascending=True)[["full_image_SSIM", "test_RMSE"]]

In [ ]:
final_metrics.sort_values("test_RMSE").groupby(["data.sparsity", "opencanopy_idx"]).head(1).groupby(["data.sparsity", "model.first_omega_0"]).size()

## Normalization experiment

In [ ]:
handpicked = np.loadtxt("data_management/normalization_low_high.csv")
no_handpicked = final_metrics[~final_metrics.opencanopy_idx.isin(handpicked)]
mean_metrics = no_handpicked.groupby(["data.normalization_scale", "model.scale"], as_index=False).mean()
scales = no_handpicked["model.scale"].sort_values(ascending=True).unique()
methods = {"Global\nmin/max": 600.0, "Global\nSTD": 85.86, "Individual\nmin/max": "individual_min_max", "Individual\nSTD": "individual_std", "No\nnormalization": 1.0}
metrics = {"RMSE": "test_RMSE", "SSIM": "full_image_SSIM", r"$R^2$": "test_R2", "ME": "test_ME"}
data = {method: {} for method in methods}

for method, method_name in methods.items():
    for metric, metric_name in metrics.items():
        data[method][metric] = mean_metrics[mean_metrics["data.normalization_scale"] == method_name].sort_values("model.scale", ascending=True)[metric_name].to_numpy()[:, np.newaxis].astype(float)
        

In [ ]:
cmaps = {"RMSE": mpl.colormaps["summer"], "SSIM": mpl.colormaps["summer"], r"$R^2$": mpl.colormaps["summer"], "ME": mpl.colormaps["RdYlGn"]}
scalar_maps = {}
for metric in metrics:
    all_data = np.hstack([data[method][metric] for method in methods])
    if metric == "ME":
        abs_min = np.abs(all_data.min())
        abs_max = np.abs(all_data.max())
        min_ = min(abs_min, abs_max)
        norm = mpl.colors.Normalize(vmin=-min_, vmax=min_)
    else:
        norm = mpl.colors.Normalize(vmin=all_data.min(), vmax=all_data.max())
    scalar_mappable = mpl.cm.ScalarMappable(norm, cmaps[metric])
    scalar_maps[metric] = scalar_mappable

# creating heatmaps manually as colorizer parameter in matplotlibs imshow doesn't seem to work
heatmaps = {method: {} for method in methods}
for method in methods:
    for metric in metrics:
        rows, cols = data[method][metric].shape
        heatmap = np.zeros((rows, cols, 4))
        for k in range(rows):
            for l in range(cols):
                heatmap[k][l] = scalar_maps[metric].to_rgba(data[method][metric][k, l])
        heatmaps[method][metric] = heatmap

fig, axes = plt.subplots(4, 6, figsize=(6.27, 8), constrained_layout=True, gridspec_kw={'width_ratios': [10, 10, 10, 10, 10, 1]})
for i, metric in enumerate(metrics):
    for j, method in enumerate(methods):
        axes[i, j].imshow(heatmaps[method][metric])
        for l in range(len(scales)):
            #axes[i, j].text(0, l, f"\\small{{{data[method][metric][l, 0].round(2)}}}", ha="center", va="center", color="white")
            axes[i, j].text(0, l, f"{data[method][metric][l, 0].round(2)}", ha="center", va="center", color="white")
        axes[i, j].set_yticks(range(len(scales)), labels=scales)
        axes[i, j].set_xticks([])
        if i == 0:
            axes[i, j].set_title(method)
        if j == 0:
            axes[i, j].set_ylabel(f"{metric}\nFF scale")
        if j == 4:
            locator = mpl.ticker.LinearLocator(3)
            cb = fig.colorbar(scalar_maps[metric], cax=axes[i,j+1], location="right", format=mpl.ticker.FormatStrFormatter('%.2f'))
            cb.locator = locator
            cb.update_ticks()

#plt.savefig("figures/target_norm_heatmap/target_norm_heatmap.pgf")

In [ ]:
# let us assume all combinations of chosen parameters are present
def make_boxplot_grid(all_results, x_var, y_var, other_params, flip_x, flip_y, metric):
    size = 4
    all_results = query_df(all_results, other_params)
    try:
        x_vals = all_results[x_var].sort_values(ascending=flip_x).unique()
    except TypeError:
        all_results[x_var] = all_results[x_var].astype(str)
        x_vals = all_results[x_var].sort_values(ascending=flip_x).unique()
    try:
        y_vals = all_results[y_var].sort_values(ascending=flip_y).unique()
    except:
        all_results[y_var] = all_results[y_var].astype(str)
        y_vals = all_results[y_var].sort_values(ascending=flip_y).unique()
    x_len = len(x_vals)
    y_len = len(y_vals)

    num_samples = len(all_results["opencanopy_idx"].unique())
    
    min_metric_val = float("inf")
    max_metric_val = float("-inf")
    for y_val in y_vals:
        for x_val in x_vals:
            query_result = query_df(all_results, {x_var: x_val, y_var: y_val})
            assert len(query_result) == num_samples, f"Not unique combination (length: {len(query_result)}) ({x_var}: {x_val}) ({y_var}: {y_val})"
            train_max = query_result[f"train_{metric}"].max()
            train_min = query_result[f"train_{metric}"].min()
            test_max = query_result[f"test_{metric}"].max()
            test_min = query_result[f"test_{metric}"].min()
            if train_max > max_metric_val:
                max_metric_val = train_max
            if train_min < min_metric_val:
                min_metric_val = train_min
            if test_max > max_metric_val:
                max_metric_val = test_max
            if test_min < min_metric_val:
                min_metric_val = test_min

    fig, ax = plt.subplots(y_len, x_len, figsize=(size*x_len, size*y_len))
    for i, y_val in enumerate(y_vals):
        for j, x_val in enumerate(x_vals):
            query_result = query_df(all_results, {x_var: x_val, y_var: y_val})
            assert len(query_result) == num_samples, f"Not unique combination (length: {len(query_result)}) ({x_var}: {x_val}) ({y_var}: {y_val})"
            #ax[i, j].boxplot(query_result[f"train_{metric}"].astype(float), label=f"train_{metric}")
            ax[i, j].boxplot(query_result[f"test_{metric}"].astype(float), label=f"test_{metric}")
            #ax[i, j].set_ylim(min_metric_val, max_metric_val)
            ax[i, j].grid()
            ax[i, j].legend()
            ax[i, j].set_title(f"{y_var}={y_val}, {x_var}={x_val}", fontsize=8)
    fig.tight_layout()
    return fig

def update_param_widgets(update):
    with out2:
        out2.clear_output(wait=True)
        left_over = list(x_var_selector.options)
        left_over.remove(x_var_selector.value)
        left_over.remove(y_var_selector.value)
        param_selectors.clear()
        for param in left_over:
            options = all_results[param].unique()
            param_selectors[param] = widgets.ToggleButtons(options=options, description=f"{param}:")
        for selector in param_selectors.values():
            display(selector)

    color_within_selector.options = [x_var_selector.value, y_var_selector.value]
        

metric_selector_grid = widgets.Dropdown(
    options=available_metrics,
    description="metric:"
)
x_var_selector = widgets.ToggleButtons(
    options=list(chosen_conf),
    description="x_var:"
)
y_var_selector = widgets.ToggleButtons(
    options=list(chosen_conf),
    description="y_var:"
)
x_flip = widgets.Checkbox(
    description="flip x-axis:"
)
y_flip = widgets.Checkbox(
    description="flip y-axis:"
)

x_var_selector.observe(update_param_widgets, names="value")
y_var_selector.observe(update_param_widgets, names="value")

display(widgets.HBox([x_var_selector, x_flip]))
display(widgets.HBox([y_var_selector, y_flip]))
display(metric_selector_grid)

param_selectors = {}

out2 = widgets.Output()
out2

In [ ]:
other_params = {name: selector.value for name, selector in param_selectors.items()}
grid_fig = make_boxplot_grid(all_results.copy(), x_var_selector.value, y_var_selector.value, other_params, x_flip.value, y_flip.value, metric_selector_grid.value)

In [ ]:
#final_metrics.groupby(["data.sparsity", "model.scale"]).mean().sort_values(["data.sparsity", "test_RMSE"])[["train_RMSE", "test_RMSE"]]
final_metrics.groupby(["data.normalization_scale", "model.scale"]).mean().sort_values(["data.normalization_scale", "test_RMSE"], ascending=True)[["full_image_SSIM", "test_RMSE"]].groupby("data.normalization_scale").head(1)[["full_image_SSIM", "test_RMSE"]]

In [ ]:
final_metrics.groupby(["data.normalization_scale", "model.scale"]).mean().sort_values(["data.normalization_scale", "test_RMSE"], ascending=True).groupby("data.normalization_scale").head(1)[["test_RMSE", "full_image_SSIM", "test_ME", "test_R2"]]

In [ ]:
final_metrics.groupby(["data.normalization_scale", "model.scale"]).mean().sort_values(["data.normalization_scale", "model.scale"], ascending=True)[["test_RMSE", "full_image_SSIM", "test_ME", "test_R2"]]

In [ ]:
final_metrics.groupby(["data.normalization_scale", "model.scale"]).std().sort_values(["data.normalization_scale", "model.scale"], ascending=True)[["test_RMSE", "full_image_SSIM", "test_ME", "test_R2"]]/np.sqrt(len(random_samples))

In [ ]:
final_metrics.sort_values(["opencanopy_idx", "test_RMSE"])[["opencanopy_idx", "model.scale", "full_image_SSIM", "test_RMSE"]]

In [ ]:
metrics = ["test_RMSE", "full_image_SSIM", "test_R2", "test_ME"]
title_map = {"test_RMSE": "RMSE", "full_image_SSIM": "SSIM", "test_R2": r"$R^2$", "test_ME": "ME"}
method_map = {1.0: "No norm", 85.86: "global std", 600.0: "global min/max", "individual_min_max": "ind min max", "individual_std": "ind std"}
random_samples_results = no_handpicked
best_scales_index = random_samples_results.groupby(["data.normalization_scale", "model.scale"]).mean().sort_values(["data.normalization_scale", "test_RMSE"], ascending=True).groupby("data.normalization_scale").head(1).index
filter_ = random_samples_results.set_index(["data.normalization_scale", "model.scale"]).index.isin(best_scales_index)

fig, axes = plt.subplots(2, 2, figsize=(6.27, 7), constrained_layout=True)
for i in range(2):
    for j in range(2):
        metric = metrics[i*2+j]
        filtered_groupby = random_samples_results[filter_][["data.normalization_scale", metric]].groupby(["data.normalization_scale"])
        final_data = [(y,x[metric].astype(float)) for y, x in filtered_groupby]
        tick_labels = [x[0][0] for x in final_data]
        box_plot_data = [x[1] for x in final_data]
        _ = axes[i,j].boxplot(box_plot_data, tick_labels=[method_map[m] for m in tick_labels], showfliers=True)
        axes[i,j].set_title(title_map[metric])
    
        mpl.artist.setp(axes[i,j].get_xticklabels(), rotation=45)

plt.savefig("figures/target_norm_boxplots/target_norm_boxplots.pgf")

In [ ]:
best_scales_index

In [ ]:
all_results[(all_results["data.normalization_scale"] == "individual_min_max") & (all_results["model.scale"] == 2)][["data.normalization_scale", "model.scale", "opencanopy_idx", "test_R2", "test_RMSE"]]

In [ ]:
all_indices = all_results.opencanopy_idx.unique()
max_heights = np.array([all_results[all_results.opencanopy_idx == idx].iloc[0]["target_image"].max() for idx in all_indices])

In [ ]:
sort_indices = np.argsort(max_heights)
max_heights = max_heights[sort_indices]
all_indices = all_indices[sort_indices]

In [ ]:
for norm_scale, model_scale in best_scales_index:
    mask = (all_results["data.normalization_scale"] == norm_scale) & (all_results["model.scale"] == model_scale)
    filtered = all_results[mask]
    metrics = [filtered[filtered.opencanopy_idx == idx]["test_RMSE"].item() for idx in all_indices]
    plt.plot(max_heights, metrics, "-", label=norm_scale)
    
plt.yscale("log")
#plt.ylim(-2, 1)
plt.legend()

# Notes / to-do

In [ ]:
# Single inspection:
# Fixed pixel ranges in single run inspection (custom ranges?)
# colorbars

# For fun:
# Make video with reconstruced images.

# Some figures

In [ ]:
fig, ax = plt.subplots(3,5,figsize=(6.27,1.9*6.27/3), constrained_layout=True)
scale_heights = 10
x_vals = [1.0, 0.5, 0.1, 0.02]
x_var = "data.sparsity"
x_var_name = "sparsity"
indices = all_results["opencanopy_idx"].unique()
idx = indices[0]
idx_rows = query_df(all_results, {"opencanopy_idx": idx})
target = idx_rows["target_image"].iloc[0] / scale_heights
min_pixel_val = target.min() # scaling is done above
max_pixel_val = target.max() # scaling is done above
ax[0, 0].imshow(target, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
ax[0, 0].set_xticks([], [])
ax[0, 0].set_yticks([], [])
ax[0, 0].set_title("Target image")
for j, x_val in enumerate(x_vals):
    query_result = query_df(idx_rows, {x_var: x_val})
    ax[0, j+1].imshow(target, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
    mask = query_result["train_mask"].iloc[0]
    print(mask.sum())
    ax[0, j+1].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
    ax[0, j+1].set_xticks([], [])
    ax[0, j+1].set_yticks([], [])
    ax[0, j+1].set_title(f"{x_var_name}={x_val}")
print()

idx = indices[1]
idx_rows = query_df(all_results, {"opencanopy_idx": idx})
target = idx_rows["target_image"].iloc[0] / scale_heights
min_pixel_val = target.min() # scaling is done above
max_pixel_val = target.max() # scaling is done above
ax[1, 0].imshow(target, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
ax[1, 0].set_xticks([], [])
ax[1, 0].set_yticks([], [])
for j, x_val in enumerate(x_vals):
    query_result = query_df(idx_rows, {x_var: x_val})
    ax[1, j+1].imshow(target, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
    mask = query_result["train_mask"].iloc[0]
    print(mask.sum())
    ax[1, j+1].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
    ax[1, j+1].set_xticks([], [])
    ax[1, j+1].set_yticks([], [])
print()

idx = indices[2]
idx_rows = query_df(all_results, {"opencanopy_idx": idx})
target = idx_rows["target_image"].iloc[0] / scale_heights
min_pixel_val = target.min() # scaling is done above
max_pixel_val = target.max() # scaling is done above
ax[2, 0].imshow(target, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
ax[2, 0].set_xticks([], [])
ax[2, 0].set_yticks([], [])
for j, x_val in enumerate(x_vals):
    query_result = query_df(idx_rows, {x_var: x_val})
    ax[2, j+1].imshow(target, vmin=min_pixel_val, vmax=max_pixel_val, cmap="summer")
    mask = query_result["train_mask"].iloc[0]
    print(mask.sum())
    ax[2, j+1].imshow(mask_to_alpha_image(mask, red=0, green=0, blue=0))
    ax[2, j+1].set_xticks([], [])
    ax[2, j+1].set_yticks([], [])
print()

plt.savefig("figures/sparsification/sparsification.pgf", dpi=250)

# Computing metrics based on images and masks

In [ ]:
from src.utils.metrics import R2, SSIM, RMSE

row = all_results[all_results["opencanopy_idx"] == 38651]
target = row.target_image.item()
recon = row.reconstructed_target.item()
test_mask = row.test_mask.item()

target_test = target[test_mask]
recon_test = recon[test_mask]

In [ ]:
rmse = RMSE()
print(rmse(recon_test.flatten(), target_test.flatten()))
print(row.test_RMSE.item())

In [ ]:
r2 = R2()
print(r2(recon_test.flatten(), target_test.flatten()))
print(row.test_R2.item())

In [ ]:
ssim = SSIM(6000, 0.01, 0.03, True, 1.5, 11, "cpu")
print(ssim(recon, target))
print(row.full_image_SSIM.item())

# Other thigns

In [ ]:
all_results["run_name"].unique()

In [ ]:
all_results["opencanopy_idx"].unique()

In [ ]:
first = all_results[(all_results.run_name == "2026-06-18T13:43:06") & (all_results.opencanopy_idx == 38651)]
second = all_results[(all_results.run_name == "2026-06-18T13:54:27") & (all_results.opencanopy_idx == 38651)]
third = all_results[(all_results.run_name == "2026-06-16T15:43:37") & (all_results.opencanopy_idx == 38651)]

In [ ]:
first

In [ ]:
second

In [ ]:
third